In [ ]:
!pip install -q netCDF4 xarray numpy matplotlib scikit-learn torch optuna plotly
!pip install --upgrade xarray scipy netCDF4 h5netcdf
import os
import random
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import warnings
from tqdm import trange
import plotly.express as px
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
import calendar
import json
from torch.optim.lr_scheduler import ReduceLROnPlateau
import plotly.subplots as sp

# --- Seed for reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# --- Environment setup ---
warnings.filterwarnings("ignore")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")

# --- Global constants ---
FILL_VAL = -999
MIN_VALID_FEATURES = 5


# ========== LOAD 10-Year Daily Chl-a Data from NetCDF ==========

netcdf_file = '/home/arnab/Downloads/CCI_ALL-v6.0-DAILY(2).nc'

# Load dataset
ds = xr.open_dataset(netcdf_file)

# Limit to desired time range (10 years)
ds = ds.sel(time=slice('2014-01-01', '2024-01-01'))

# Remove duplicate lat/lon entries (if any)
lat_unique, lat_idx = np.unique(ds['lat'].values, return_index=True)
lon_unique, lon_idx = np.unique(ds['lon'].values, return_index=True)
ds = ds.isel(lat=np.sort(lat_idx), lon=np.sort(lon_idx))

# Extract chlorophyll variable
chl = ds['chlor_a']

# Confirm load
print("✅ Dataset loaded:")
print(f"Time range : {str(ds['time'].values[0])[:10]} to {str(ds['time'].values[-1])[:10]}")
print(f"Time steps : {ds.sizes['time']} | Lat: {ds.sizes['lat']} | Lon: {ds.sizes['lon']}")

#========== RAW STATISTICS (Before Any Cleaning, 10 Years) ==========

chl_array = chl.values
valid_chl = chl_array[~np.isnan(chl_array)]

# Grid + time dimensions
n_days = chl.sizes['time']
n_lat = chl.sizes['lat']
n_lon = chl.sizes['lon']
grid_pixels = n_lat * n_lon
total_points = chl_array.size

# Missing statistics
missing_points = np.isnan(chl_array).sum()
observed_points = total_points - missing_points
pct_missing = 100 * missing_points / total_points if total_points else 0

# Permanently missing grid points
always_missing_pix = int(chl.isnull().all(dim='time').sum().item())
effective_pixels = grid_pixels - always_missing_pix

# Basic Chl-a stats
min_val = np.nanmin(valid_chl) if valid_chl.size else np.nan
max_val = np.nanmax(valid_chl) if valid_chl.size else np.nan
mean_val = np.nanmean(valid_chl) if valid_chl.size else np.nan
std_val = np.nanstd(valid_chl) if valid_chl.size else np.nan

# Summary
print("=== Raw Chl-a Statistics (Before Cleaning, 10 Years) ===")
print(f"Total days             : {n_days}")
print(f"Lat × Lon              : {n_lat} × {n_lon}")
print(f"Grid pixels            : {grid_pixels}")
print(f"• Permanently missing  : {always_missing_pix}")
print(f"• Effective pixels     : {effective_pixels}")
print(f"Total data points      : {total_points}")
print(f"Observed points        : {observed_points}")
print(f"Missing points         : {missing_points} ({pct_missing:.2f}%)")
print(f"Chl-a min (obs)        : {min_val:.3f}")
print(f"Chl-a max (obs)        : {max_val:.3f}")
print(f"Chl-a mean (obs)       : {mean_val:.3f}")
print(f"Chl-a std dev (obs)    : {std_val:.3f}")






# ========== DATA CLEANING (Log10 + 3σ Clipping) ==========

# Step 1: Domain-based physical filtering (before log)
CHL_MIN = 0.01   # To avoid log(0) and negative values
CHL_MAX = 50.0   # You may adjust this for specific region sensitivity

chl = chl.where((chl >= CHL_MIN) & (chl <= CHL_MAX))

# Step 2: Log10 transform
chl_log = np.log10(chl)

# Step 3: 3σ outlier clipping in log space
chl_log_vals = chl_log.values
chl_log_valid = chl_log_vals[~np.isnan(chl_log_vals)]

log_mean = np.nanmean(chl_log_valid)
log_std  = np.nanstd(chl_log_valid)
log_clip_min = log_mean - 3 * log_std
log_clip_max = log_mean + 3 * log_std

chl_log_clipped = chl_log.clip(min=log_clip_min, max=log_clip_max)

# Step 4: Inverse transform back to linear space (mg/m³)
chl_clean = xr.apply_ufunc(
    lambda x: 10 ** x,
    chl_log_clipped,
    dask="allowed"
)

# Step 5: Generate updated missing mask
mask = xr.DataArray(
    np.where(np.isnan(chl_clean), 1, 0),
    dims=chl_clean.dims,
    coords=chl_clean.coords
)

# Summary logs
print("\n✅ Data cleaning complete:")
print(f"• Physical range filtered  : [{CHL_MIN}, {CHL_MAX}] mg/m³")
print(f"• Log10 mean ± 3σ clipping : [{log_clip_min:.3f}, {log_clip_max:.3f}]")
print(f"• Linear scale after clip  : [{10**log_clip_min:.3f}, {10**log_clip_max:.3f}]")

# ========== CLEANED CHL-a STATISTICS ==========

chl_vals_clean = chl_clean.values
valid_vals = chl_vals_clean[~np.isnan(chl_vals_clean)]

n_days = chl_clean.sizes['time']
n_lat  = chl_clean.sizes['lat']
n_lon  = chl_clean.sizes['lon']
total_points = chl_vals_clean.size

missing_points = np.isnan(chl_vals_clean).sum()
observed_points = total_points - missing_points
pct_missing = 100 * missing_points / total_points if total_points else 0

n_permanent_missing = int(chl_clean.isnull().all(dim='time').sum().item())
n_effective_pixels  = n_lat * n_lon - n_permanent_missing

min_val = np.nanmin(valid_vals) if valid_vals.size else np.nan
max_val = np.nanmax(valid_vals) if valid_vals.size else np.nan
mean_val = np.nanmean(valid_vals) if valid_vals.size else np.nan
std_val  = np.nanstd(valid_vals) if valid_vals.size else np.nan

print("=== Cleaned Chl-a Statistics (mg/m³) ===")
print(f"Time steps             : {n_days}")
print(f"Lat × Lon              : {n_lat} × {n_lon}")
print(f"Grid pixels            : {n_lat * n_lon}")
print(f"• Permanently missing  : {n_permanent_missing}")
print(f"• Effective pixels     : {n_effective_pixels}")
print(f"Total data points      : {total_points}")
print(f"Missing points         : {missing_points} ({pct_missing:.2f}%)")
print(f"Observed points        : {observed_points}")
print(f"Chl-a min (obs)        : {min_val:.3f} mg/m³")
print(f"Chl-a max (obs)        : {max_val:.3f} mg/m³")
print(f"Chl-a mean (obs)       : {mean_val:.3f} mg/m³")
print(f"Chl-a std dev (obs)    : {std_val:.3f} mg/m³")



custom_scale = [
    [0.0, 'blue'],      # Low values
    [0.25, 'cyan'],     # Light blue
    [0.5, 'yellow'],    # Mid values
    [0.75, 'orange'],   # Higher values
    [1.0, 'red']        # Highest values
]







# --- 1. Get all days with ~40–50% observed data ---
all_times = pd.to_datetime(chl_clean['time'].values)

# Calculate observation percentage for each day
obs_ratio = {}
for t in all_times:
    day_data = chl_clean.sel(time=t).values
    total_points = day_data.size
    valid_points = np.count_nonzero(~np.isnan(day_data))
    obs_ratio[t] = (valid_points / total_points) * 100  # percentage

# Filter days with 40–50% observed data
filtered_days = [t for t, pct in obs_ratio.items() if 40 <= pct <= 50]

# Randomly choose 3 days from the filtered list, fallback to other gap days if not enough
np.random.seed(42)
if len(filtered_days) >= 3:
    chosen_days = np.random.choice(filtered_days, 3, replace=False)
elif len(filtered_days) > 0:
    chosen_days = np.random.choice(filtered_days, min(3, len(filtered_days)), replace=False)
else:
    days_with_gaps = [t for t in all_times if np.isnan(chl_clean.sel(time=t).values).any()]
    chosen_days = np.random.choice(days_with_gaps, 3, replace=False)

# --- 2. Create interactive Plotly figure ---
fig = sp.make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"Observed Chl-a<br>{str(day)[:10]} ({obs_ratio[day]:.1f}% obs)" for day in chosen_days],
    shared_yaxes=True
)

for i, day in enumerate(chosen_days, 1):
    data = chl_clean.sel(time=day).values

    fig.add_trace(go.Contour(
        z=data,
        x=chl_clean['lon'].values,
        y=chl_clean['lat'].values,
        colorscale=custom_scale,
        colorbar=dict( len=0.8, x=1.05 if i == 3 else None),
        contours=dict(showlines=False),
        #hovertemplate="Lat: %{y:.2f}<br>Lon: %{x:.2f}<br>Chl-a: %{z:.3f} mg/m³<extra></extra>",
        zmin=np.nanmin(chl_clean.values),
        zmax=np.nanmax(chl_clean.values)
    ), row=1, col=i)

fig.update_layout(
    height=400,
    width=1100,
    title_text="🌊 Observed Chlorophyll-a (40–50% Observed Data) - 3 Random Days",
    template="plotly_white"
)
fig.update_xaxes(title_text="Longitude")
fig.update_yaxes(title_text="Latitude")

fig.show()






# ========== INTERACTIVE EXPLORATORY PLOTS (Cleaned Chl-a) ==========

# --- 1. Time Series of Spatially Averaged Chl-a ---
ts_mean = chl_clean.mean(dim=['lat', 'lon'], skipna=True)

fig = px.line(
    x=ts_mean['time'].values,
    y=ts_mean.values,
    labels={'x': 'Date', 'y': 'Mean Chl-a (mg/m³)'},
    title='📈 Spatially Averaged Chlorophyll-a Over Time (10 Years)'
)
fig.update_traces(mode='lines+markers')
fig.update_layout(template="plotly_white")
fig.show()

# --- 2. Seasonal Grouping (DJF, MAM, JJA, SON) ---

# Helper: Season mapper
def season_of(month):
    return ('DJF', 'MAM', 'JJA', 'SON')[(month % 12) // 3]

# Create DataFrame with seasonal labels
season_df = (
    pd.Series(ts_mean.values, index=pd.to_datetime(ts_mean['time'].values))
    .to_frame('mean')
)
season_df['season'] = season_df.index.month.map(season_of)

# Grouped seasonal means
season_means = season_df.groupby('season')['mean'].mean()
print(f"🌤️ Season with highest mean : {season_means.idxmax()} ({season_means.max():.3f} mg/m³)")
print(f"❄️  Season with lowest mean  : {season_means.idxmin()} ({season_means.min():.3f} mg/m³)\n")

# Plot seasonal time series
fig_season = px.line(
    season_df.reset_index(),
    x='index', y='mean', color='season',
    labels={'index': 'Date', 'mean': 'Mean Chl-a (mg/m³)', 'season': 'Season'},
    title='🌿 Seasonal Trends in Chlorophyll-a (5 Years)'
)
fig_season.update_layout(template="plotly_white")
fig_season.show()

# --- 3. Monthly Mean Bar Plot ---

# Group by month
monthly_avg = ts_mean.groupby(ts_mean['time.month']).mean()
months = np.arange(1, 13)
m_max = months[monthly_avg.values.argmax()]
m_min = months[monthly_avg.values.argmin()]

fig_month = px.bar(
    x=months, y=monthly_avg.values,
    labels={'x': 'Month', 'y': 'Mean Chl-a (mg/m³)'},
    title='📊 Monthly Mean Chlorophyll-a (10-Year Average)'
)
fig_month.add_vline(x=m_max, line_dash='dash', annotation_text='Max', annotation_position='top')
fig_month.add_vline(x=m_min, line_dash='dash', annotation_text='Min', annotation_position='bottom')
fig_month.update_layout(template="plotly_white")
fig_month.show()

print(f"📅 Month with highest mean : {calendar.month_name[m_max]}")
print(f"📅 Month with lowest mean  : {calendar.month_name[m_min]}")


# ========== FEATURE ENGINEERING (10-Year Compatible, Multi-Channel Spatial Patch, for CNN-MLP-MLP Model) ==========

# --- 1. BASE: Use cleaned chl-a
chl_base = chl_clean.copy()

# --- 2. Spatial Context: 3×3 Grid (center pixel + 8 neighbors) ---
window = 3
d = window // 2

# -- A. Chl-a 3x3 patches
chl_pad = chl_base.pad(lat=d, lon=d, mode='constant', constant_values=np.nan)
sp_vars = []
for i in range(window):
    for j in range(window):
        v = chl_pad.isel(lat=slice(i, i + chl_base.sizes['lat']),
                         lon=slice(j, j + chl_base.sizes['lon']))
        v = v.assign_coords(lat=chl_base['lat'], lon=chl_base['lon']).rename(f'nb_{i}_{j}')
        sp_vars.append(v)

# -- B. Mask 3x3 patches
mask_pad = mask.pad(lat=d, lon=d, mode='constant', constant_values=1)
mask_vars = []
for i in range(window):
    for j in range(window):
        mv = mask_pad.isel(lat=slice(i, i + mask.sizes['lat']),
                           lon=slice(j, j + mask.sizes['lon']))
        mv = mv.assign_coords(lat=chl_base['lat'], lon=chl_base['lon']).rename(f'nbmask_{i}_{j}')
        mask_vars.append(mv)

# --- 3. Mean Annual Climatology Map (3x3 patches) ---
annual_mean = chl_base.mean(dim='time', skipna=True)
annual_map = annual_mean.broadcast_like(chl_base).rename('chl_ann_mean')
clim_pad = annual_map.pad(lat=d, lon=d, mode='constant', constant_values=np.nan)
clim_vars = []
for i in range(window):
    for j in range(window):
        v = clim_pad.isel(lat=slice(i, i + chl_base.sizes['lat']),
                          lon=slice(j, j + chl_base.sizes['lon']))
        v = v.assign_coords(lat=chl_base['lat'], lon=chl_base['lon']).rename(f'clim_{i}_{j}')
        clim_vars.append(v)

# --- 4. Seasonal Mean Maps (3x3 patches for each) ---
def get_season_mask(ds, season_name):
    month_to_season = {
        12: 'DJF', 1: 'DJF', 2: 'DJF',
        3: 'MAM', 4: 'MAM', 5: 'MAM',
        6: 'JJA', 7: 'JJA', 8: 'JJA',
        9: 'SON', 10: 'SON', 11: 'SON'
    }
    season_months = [m for m in range(1, 13) if month_to_season[m] == season_name]
    return ds['time.month'].isin(season_months)

season_names = ['DJF', 'MAM', 'JJA', 'SON']
seasonal_patch_vars = {s: [] for s in season_names}
for season in season_names:
    mask_season = get_season_mask(chl_base, season)
    season_mean = chl_base.sel(time=chl_base['time'][mask_season]).mean(dim='time', skipna=True)
    season_map = season_mean.broadcast_like(chl_base).rename(f'chl_season_{season}')
    pad_map = season_map.pad(lat=d, lon=d, mode='constant', constant_values=np.nan)
    for i in range(window):
        for j in range(window):
            v = pad_map.isel(lat=slice(i, i + chl_base.sizes['lat']),
                             lon=slice(j, j + chl_base.sizes['lon']))
            v = v.assign_coords(lat=chl_base['lat'], lon=chl_base['lon']).rename(f'{season.lower()}_{i}_{j}')
            seasonal_patch_vars[season].append(v)

# --- 5. Lat-Lon Normalization ---
lat_norm = ((chl_base['lat'] - chl_base['lat'].min()) /
            (chl_base['lat'].max() - chl_base['lat'].min())).broadcast_like(chl_base).rename('lat_norm')
lon_norm = ((chl_base['lon'] - chl_base['lon'].min()) /
            (chl_base['lon'] - chl_base['lon'].min())).broadcast_like(chl_base).rename('lon_norm')

# --- 6. Temporal Lag Features ---
lags = [1, 2, 3, 7, 14, 30, 90, 180]
lag_feats = {f'chl_t{l}': chl_base.shift(time=l) for l in lags}
for k in list(lag_feats.keys()):
    lag_feats[k] = lag_feats[k].rename(k)

# --- 7. Temporal Rolling Means ---
rolls = [7, 14, 30, 90, 180]
roll_feats = {f'chl_roll{r}': chl_base.rolling(time=r, min_periods=1).mean() for r in rolls}
for k in list(roll_feats.keys()):
    roll_feats[k] = roll_feats[k].rename(k)

# --- 8. Temporal Seasonal Encoding (sin/cos DOY) ---
doy = chl_base['time'].dt.dayofyear
season_sin = xr.DataArray(np.sin(2 * np.pi * doy / 365), coords=[chl_base['time']], dims=['time']).broadcast_like(chl_base).rename('season_sin')
season_cos = xr.DataArray(np.cos(2 * np.pi * doy / 365), coords=[chl_base['time']], dims=['time']).broadcast_like(chl_base).rename('season_cos')

# --- 9. Combine All Features (all 3x3 patches as channels!) ---
all_patch_vars = sp_vars + mask_vars + clim_vars
for s in season_names:
    all_patch_vars += seasonal_patch_vars[s]

dsf = xr.merge(
    all_patch_vars +
    [lat_norm, lon_norm] +
    list(lag_feats.values()) + list(roll_feats.values()) +
    [season_sin, season_cos]
)

# --- 10. Flatten to DataFrame ---
df = dsf.to_dataframe().reset_index()

# --- 11. Filter Always-Missing Pixels ---
vc = chl_base.count(dim='time').to_dataframe('count_valid').reset_index()
vc['always_missing'] = vc['count_valid'] == 0
df = df.merge(vc[['lat', 'lon', 'always_missing']], on=['lat', 'lon'])
df = df[df['always_missing'] == False].drop(columns='always_missing')

# --- 12. Merge Back Mask and Target Chl-a ---
df = (df
      .merge(mask.to_dataframe('mask').reset_index(), on=['time', 'lat', 'lon'])
      .merge(chl_base.to_dataframe('chl_orig').reset_index(), on=['time', 'lat', 'lon'])
)

# --- 13. Define Feature Columns (include new patch names) ---
nb_cols = [c for c in df.columns if c.startswith('nb_') and not c.endswith('_isna')]
nbmask_cols = [c for c in df.columns if c.startswith('nbmask_')]
clim_patch_cols = [c for c in df.columns if c.startswith('clim_')]
season_patch_cols = []
for s in ['djf', 'mam', 'jja', 'son']:
    season_patch_cols += [c for c in df.columns if c.startswith(f'{s}_')]
lag_cols = [c for c in df.columns if c.startswith('chl_t')]
roll_cols = [c for c in df.columns if c.startswith('chl_roll')]
spatial_static_cols = ['lat_norm', 'lon_norm', 'nbr_valid_count']
temporal_static_cols = ['season_sin', 'season_cos']

# --- 14. Add Valid Neighbor Count ---
df['nbr_valid_count'] = (df[nbmask_cols] == 0).sum(axis=1)

# --- 15. Add NaN Flags for Missing Temporal Values ---
temporal_cols_raw = lag_cols + roll_cols
for col in temporal_cols_raw:
    mean_val = df.groupby('time')[col].transform('mean')
    df[col] = df[col].fillna(mean_val).fillna(FILL_VAL)
    df[col + '_isna'] = (df[col] == FILL_VAL).astype(np.float32)

# --- 16. Add NaN Flags for Spatial Neighbors ---
for col in nb_cols:
    mean_val = df.groupby(['lat', 'lon'])[col].transform('mean')
    df[col] = df[col].fillna(mean_val).fillna(FILL_VAL)
    df[col + '_isna'] = (df[col] == FILL_VAL).astype(np.float32)

# --- 17. Normalize Temporal Features ---
from sklearn.preprocessing import StandardScaler
temp_scl = StandardScaler().fit(df.loc[df['mask'] == 0, temporal_cols_raw].replace(FILL_VAL, np.nan).dropna())
df[temporal_cols_raw] = temp_scl.transform(df[temporal_cols_raw])

# --- 18. Normalize Target ---
target_scl = StandardScaler().fit(df.loc[df['mask'] == 0, ['chl_orig']].values)
df['chl_scaled'] = np.nan
df.loc[df['mask'] == 0, 'chl_scaled'] = target_scl.transform(df.loc[df['mask'] == 0, ['chl_orig']].values).ravel()

# --- 19. Final Column Groups (spatial includes all patch cols) ---

# --- 1. Identify missing indicator columns
t_mask_cols = [c for c in df.columns if c.endswith('_isna') and c.startswith('chl_')]
nb_mask_cols = [c for c in df.columns if c.endswith('_isna') and c.startswith('nb_')]

# --- 2. Explicitly separate scalar features
extra_scalar_cols = ['lat_norm', 'lon_norm', 'nbr_valid_count']

# --- ✅ 3. Spatial CNN input → strictly patch features (3x3 × channels)
spatial_cols = (
    nb_cols +         # 9 neighborhood chl-a patches
    nb_mask_cols +    # 9 neighborhood mask patches
    clim_patch_cols + # 9 climate normal patches
    season_patch_cols # 9 seasonal patches
)

# --- ✅ 4. Temporal+MLP input → time-series + extra scalars
temporal_cols = (
    lag_cols +              # 7-day chl-a lag
    roll_cols +             # rolling stats
    t_mask_cols +           # lag mask indicators
    temporal_static_cols +  # doy, season, etc.
    extra_scalar_cols       # scalar spatial (lat/lon/nbr_valid_count)
)

# --- 5. All input features
feature_cols = spatial_cols + temporal_cols
target_col = 'chl_scaled'



# ========== 3. DATA SPLIT (Spatio-Temporal Holdout, No CV) ==========

# Ensure datetime and season labels
df['time'] = pd.to_datetime(df['time'])
df['month'] = df['time'].dt.month
df['season'] = df['month'].map(lambda m: ('DJF', 'MAM', 'JJA', 'SON')[(m % 12) // 3])

# ----- Step 1: Fixed Test Set (Spatio-Temporal Holdout) -----
lat_thresh = df['lat'].quantile(0.25)  # Southern 25% of latitudes
region_mask = df['lat'] <= lat_thresh
season_mask = df['season'] == 'JJA'    # Monsoon season

df['test_flag'] = region_mask & season_mask

test_df = df[
    (df['test_flag']) &
    (df['mask'] == 0) &
    (df['nbr_valid_count'] <= 4)
].dropna(subset=feature_cols + ['chl_scaled']).copy()

# ----- Step 2: Train/Val Split on Remaining (Simple Random Split) -----
trainval_df = df[~df['test_flag'] & (df['mask'] == 0)].dropna(subset=feature_cols + ['chl_scaled']).copy()
n_total = len(trainval_df)
n_train = int(0.85 * n_total)

train_df = trainval_df.iloc[:n_train].copy()
val_df   = trainval_df.iloc[n_train:].copy()

print(f"✅ Train Set Samples : {len(train_df)}")
print(f"🧪 Val Set Samples   : {len(val_df)}")
print(f"🧾 Test Set Samples  : {len(test_df)}")

# ----- Step 3: Finalize Holdout/Test Flag -----
df['mask_holdout'] = False
df.loc[test_df.index, 'mask_holdout'] = True

# Clean-up temp cols
df.drop(columns=['test_flag', 'month', 'season'], inplace=True, errors='ignore')

# Summary
print("\n✅ Spatio-Temporal Holdout Split Done (No CV)")
print(f"Test Region: Southern 25% (lat ≤ {lat_thresh:.2f}) | Season: JJA (Monsoon)")
print(f"Test Days: {test_df['time'].nunique()} | Test Locations: {test_df[['lat','lon']].drop_duplicates().shape[0]}")

# ========== 4. DATASET CLASS FOR CNN+MLP+MLP HYBRID MODEL  ==========
class ChlCNNMLPTemporalMLPFusionDataset(Dataset):
    def __init__(self, df, spat_cols, temp_cols, target_col,
                 augment=False, aug_prob=0.1):
        self.df = df.reset_index(drop=True)
        self.spat_cols = spat_cols
        self.temp_cols = temp_cols
        self.target_col = target_col
        self.augment = augment
        self.aug_prob = aug_prob

        # ✅ Prepare temporal features (includes lat/lon scalars now)
        self.temp_data = df[temp_cols].values.astype(np.float32)

        # ✅ Prepare spatial patch tensor [N, C, 3, 3]
        self.spatial_images = self._build_spatial_tensor(df[spat_cols])

        # Target variable
        self.target_data = df[target_col].values.astype(np.float32)

    def _build_spatial_tensor(self, df_spatial):
        N = len(df_spatial)
        num_channels = len(self.spat_cols) // 9
        assert len(self.spat_cols) % 9 == 0, \
            f"❌ Spatial features ({len(self.spat_cols)}) not divisible by 9!"
        img_tensor = df_spatial[self.spat_cols].values.astype(np.float32)
        img_tensor = img_tensor.reshape(N, num_channels, 3, 3)
        return img_tensor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        x_spat = self.spatial_images[idx]        # [C, 3, 3]
        x_temp = self.temp_data[idx]             # [T] (includes scalars)
        y = self.target_data[idx]

        # ✅ Optional random augmentation
        if self.augment:
            if random.random() < self.aug_prob:
                x_spat = x_spat.copy()
                mask_channel = np.random.rand(*x_spat[0].shape) < 0.1
                x_spat[0][mask_channel] = FILL_VAL
                x_spat[1][mask_channel] = 1.0

            if random.random() < self.aug_prob:
                x_temp = x_temp.copy()
                temp_mask = np.random.rand(*x_temp.shape) < 0.1
                x_temp[temp_mask] = FILL_VAL

        return (
            torch.tensor(x_spat, dtype=torch.float32),   # [C, 3, 3]
            torch.tensor(x_temp, dtype=torch.float32),   # [T]
            torch.tensor(y, dtype=torch.float32)
        )

#  ========== 5. MODEL: CNN (Spatial) + MLP (Temporal) + MLP (Fusion) ==========

class CNNSpatial_MLPTemporal_MLPFusion(nn.Module):
    def __init__(self, n_temp_features, h_c=128, drop=0.2):
        super().__init__()

        # --- Spatial CNN branch (same as before) ---
        self.cnn_spatial = nn.Sequential(
            nn.Conv2d(in_channels=7, out_channels=16, kernel_size=2),  # (16, 2, 2)
            nn.SiLU(),
            nn.Flatten(),           # → 16 * 2 * 2 = 64
            nn.Dropout(drop)
        )

        # --- Temporal MLP branch (instead of 1D CNN) ---
        self.mlp_temporal = nn.Sequential(
            nn.Linear(n_temp_features, 64),
            nn.SiLU(),
            nn.Dropout(drop),
            nn.Linear(64, 32),
            nn.SiLU(),
            nn.Dropout(drop)
        )

        # --- Fusion MLP (same as before) ---
        self.fusion_net = nn.Sequential(
            nn.Linear(64 + 32, h_c),
            nn.SiLU(),
            nn.Dropout(drop),
            nn.Linear(h_c, 1)
        )

    def forward(self, x_spat, x_temp):
        spat_feat = self.cnn_spatial(x_spat)           # [B, 64]
        temp_feat = self.mlp_temporal(x_temp)           # [B, 32]  # <-- x_temp shape: [B, T]
        fusion_in = torch.cat([spat_feat, temp_feat], dim=1)  # [B, 96]
        return self.fusion_net(fusion_in).squeeze(1)          # [B]


# ========== 6. TRAINING FUNCTION (Mixed Precision, Early Stop, Warm-up LR) ==========
def train_cnn_mlp_mlp(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    n_epochs=120,
    n_warmup=50,
    patience=10,
    device='cuda',
    verbose=True,
):




    model.to(device)
    best_val_loss = float('inf')
    best_state_dict = None
    train_losses = []
    val_losses = []
    patience_counter = 0

    for epoch in range(1, n_epochs + 1):
        # --- Training ---
        model.train()
        train_loss = 0.0
        n_train = 0
        for x_spat, x_temp, y in train_loader:
            x_spat, x_temp, y = x_spat.to(device), x_temp.to(device), y.to(device)
            optimizer.zero_grad()
            y_pred = model(x_spat, x_temp)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * y.size(0)
            n_train += y.size(0)
        train_loss /= n_train
        train_losses.append(train_loss)

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for x_spat, x_temp, y in val_loader:
                x_spat, x_temp, y = x_spat.to(device), x_temp.to(device), y.to(device)
                y_pred = model(x_spat, x_temp)
                loss = criterion(y_pred, y)
                val_loss += loss.item() * y.size(0)
                n_val += y.size(0)
        val_loss /= n_val
        val_losses.append(val_loss)
        scheduler.step(val_loss)
        current_lr= optimizer.param_groups[0]['lr']

        # --- Print ---
        if verbose:
            print(f"Epoch {epoch:03d}/{n_epochs} | "
                  f"Train Loss: {train_loss:.5f} |"
                  f"Val Loss: {val_loss:.5f}|"
                  f"LR:{current_lr:.6f}", end='')

        # --- Early Stopping Logic ---
        if epoch > n_warmup:
            if val_loss < best_val_loss - 1e-6:  # Significant improvement
                best_val_loss = val_loss
                best_state_dict = model.state_dict()
                patience_counter = 0
                if verbose: print("  [*] Best val loss updated!")
            else:
                patience_counter += 1
                if verbose: print(f"  [Patience: {patience_counter}/{patience}]")
                if patience_counter >= patience:
                    if verbose: print("Early stopping triggered. Restoring best model.")
                    break
        else:
            if verbose: print("  [Warmup]")

    # Restore best model weights
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

# ========== 7. HYPERPARAMETERS (CNN + CNN + MLP Hybrid) (No CV, No Optuna) ==========

HYPERPARAMS = dict(
    # --- Model ---
    h_c = 128,                  # Hidden size in fusion MLP
    drop = 0.35,                 # Dropout rate

    # --- Training ---
    batch_size = 64,
    lr = 5e-4,                  # Learning rate
    weight_decay = 1e-3,        # L2 regularization
    n_epochs = 120,
    n_warmup = 50,              # Warmup epochs (no early stopping)
    patience = 10,              # Early stopping patience

    # --- Augmentation ---
    augment = True,             # Enable random mask/temporal drop
    aug_prob = 0.25,            # Probability of applying augmentation

    # --- Others ---
    device = device,            # 'cuda' or 'cpu'
    random_seed = SEED,
)

# For reproducibility
torch.manual_seed(HYPERPARAMS['random_seed'])
np.random.seed(HYPERPARAMS['random_seed'])
random.seed(HYPERPARAMS['random_seed'])

# For DataLoader
BATCH_SIZE = HYPERPARAMS['batch_size']



# ========== 8. TRAINING THE MODEL  ==========
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# --- Build datasets and loaders ---
train_dataset = ChlCNNMLPTemporalMLPFusionDataset(
    train_df, spatial_cols, temporal_cols, target_col,
    augment=HYPERPARAMS['augment'], aug_prob=HYPERPARAMS['aug_prob']
)
val_dataset = ChlCNNMLPTemporalMLPFusionDataset(
    val_df, spatial_cols, temporal_cols, target_col,
    augment=False
)
train_loader = DataLoader(train_dataset, batch_size=HYPERPARAMS['batch_size'], shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=HYPERPARAMS['batch_size'], shuffle=False, drop_last=False)

# --- Initialize model ---
model = CNNSpatial_MLPTemporal_MLPFusion(
    n_temp_features=len(temporal_cols),
    h_c=HYPERPARAMS['h_c'],
    drop=HYPERPARAMS['drop']
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=HYPERPARAMS['lr'],
    weight_decay=HYPERPARAMS['weight_decay']
)
criterion = nn.MSELoss()

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',        # We want to reduce LR when val_loss stops decreasing
    factor=0.5,        # Reduce LR by a factor of 0.5
    patience=5,        # Number of epochs to wait before reducing LR
    min_lr=1e-6,       # Lower bound on learning rate
    verbose=True       # Prints LR updates
)


# --- Train ---
model, train_losses, val_losses = train_cnn_mlp_mlp(
    model, train_loader, val_loader, optimizer, criterion,
    n_epochs=HYPERPARAMS['n_epochs'],
    n_warmup=HYPERPARAMS['n_warmup'],
    patience=HYPERPARAMS['patience'],
    device=HYPERPARAMS['device'],
    verbose=True
)

# --- Print epoch-wise losses ---
print("\n📉 Epoch-wise Train and Val Losses:")
for epoch, (tr_l, val_l) in enumerate(zip(train_losses, val_losses), 1):
    print(f"Epoch {epoch:03d} | Train Loss: {tr_l:.5f} | Val Loss: {val_l:.5f}")

# --- Save model ---
saved_model_path = "cnn_mlp_mlp_final.pt"
torch.save(model.state_dict(), saved_model_path)

# --- Inverse Transform Train Predictions ---
y_true_train_unscaled = target_scl.inverse_transform(np.array(train_df[target_col]).reshape(-1, 1)).ravel()

train_dataset_noscale = ChlCNNMLPTemporalMLPFusionDataset(
    train_df, spatial_cols, temporal_cols, target_col,
    augment=False
)
train_loader_noscale = DataLoader(train_dataset_noscale, batch_size=HYPERPARAMS['batch_size'], shuffle=False)

model.eval()
y_pred_train_scaled = []
with torch.no_grad():
    for x_spat, x_temp, _ in train_loader_noscale:
        x_spat, x_temp = x_spat.to(HYPERPARAMS['device']), x_temp.to(HYPERPARAMS['device'])
        y_pred = model(x_spat, x_temp)
        y_pred_train_scaled.append(y_pred.cpu().numpy())
y_pred_train_scaled = np.concatenate(y_pred_train_scaled)
y_pred_train_unscaled = target_scl.inverse_transform(y_pred_train_scaled.reshape(-1, 1)).ravel()

# --- Metrics on Unscaled Train ---
rmse_train_unscaled = np.sqrt(mean_squared_error(y_true_train_unscaled, y_pred_train_unscaled))
mae_train_unscaled = mean_absolute_error(y_true_train_unscaled, y_pred_train_unscaled)
r2_train_unscaled = r2_score(y_true_train_unscaled, y_pred_train_unscaled)

print("\n🎓 Train Metrics (Original Units mg/m³):")
print(f"RMSE: {rmse_train_unscaled:.4f} mg/m³")
print(f"MAE : {mae_train_unscaled:.4f} mg/m³")
print(f"R²  : {r2_train_unscaled:.4f}")

# --- Inverse Transform Val Predictions ---
y_true_val_unscaled = target_scl.inverse_transform(np.array(val_df[target_col]).reshape(-1, 1)).ravel()

val_dataset_noscale = ChlCNNMLPTemporalMLPFusionDataset(
    val_df, spatial_cols, temporal_cols, target_col,
    augment=False
)
val_loader_noscale = DataLoader(val_dataset_noscale, batch_size=HYPERPARAMS['batch_size'], shuffle=False)

y_pred_val_scaled = []
with torch.no_grad():
    for x_spat, x_temp, _ in val_loader_noscale:
        x_spat, x_temp = x_spat.to(HYPERPARAMS['device']), x_temp.to(HYPERPARAMS['device'])
        y_pred = model(x_spat, x_temp)
        y_pred_val_scaled.append(y_pred.cpu().numpy())
y_pred_val_scaled = np.concatenate(y_pred_val_scaled)
y_pred_val_unscaled = target_scl.inverse_transform(y_pred_val_scaled.reshape(-1, 1)).ravel()

# --- Metrics on Unscaled Validation ---
rmse_val_unscaled = np.sqrt(mean_squared_error(y_true_val_unscaled, y_pred_val_unscaled))
mae_val_unscaled = mean_absolute_error(y_true_val_unscaled, y_pred_val_unscaled)
r2_val_unscaled = r2_score(y_true_val_unscaled, y_pred_val_unscaled)

print("\n🧪 Validation Metrics (Original Units mg/m³):")
print(f"RMSE: {rmse_val_unscaled:.4f} mg/m³")
print(f"MAE : {mae_val_unscaled:.4f} mg/m³")
print(f"R²  : {r2_val_unscaled:.4f}")



# ========== 9. TRAINING SET EVALUATION ==========

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import torch

# --- 1. Plot Train/Val Loss Curves (Single Run) ---
epochs = np.arange(1, len(train_losses) + 1)

plt.figure(figsize=(7,4))
plt.plot(epochs, train_losses, label='Train Loss', color='blue')
plt.plot(epochs, val_losses, label='Val Loss', color='orange')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.ylim(0, 0.15)
plt.title('Train vs. Validation Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# --- 2. Print Final Train/Val Metrics ---
train_metrics = [rmse_train_unscaled, mae_train_unscaled, r2_train_unscaled]
val_metrics   = [rmse_val_unscaled,  mae_val_unscaled,  r2_val_unscaled]

metrics_names = ['RMSE', 'MAE', 'R²']
print("\n========== Final Training Set Metrics ==========")
for i, name in enumerate(metrics_names):
    print(f"{name:>4s} | Train: {train_metrics[i]:.4f} | Val: {val_metrics[i]:.4f}")

# --- 3. Compute Train/Val Metrics (RMSE, MAE, R²) on Unscaled Data ---

# ========== Train Unscaled Metrics ==========
y_true_train_scaled = np.array(train_df[target_col].values).reshape(-1, 1)

# DataLoader (uses 7 spatial channels from spatial_cols)
train_dataset_noscale = ChlCNNMLPTemporalMLPFusionDataset(
    train_df, spatial_cols, temporal_cols, target_col, augment=False
)
train_loader_noscale = DataLoader(train_dataset_noscale, batch_size=HYPERPARAMS['batch_size'], shuffle=False)

model.eval()
y_pred_train_scaled = []
with torch.no_grad():
    for x_spat, x_temp, _ in train_loader_noscale:
        x_spat, x_temp = x_spat.to(HYPERPARAMS['device']), x_temp.to(HYPERPARAMS['device'])
        y_pred = model(x_spat, x_temp)
        y_pred_train_scaled.append(y_pred.cpu().numpy())
y_pred_train_scaled = np.concatenate(y_pred_train_scaled)

# Inverse transform
y_true_train_unscaled = target_scl.inverse_transform(y_true_train_scaled).ravel()
y_pred_train_unscaled = target_scl.inverse_transform(y_pred_train_scaled.reshape(-1, 1)).ravel()

rmse_train_unscaled = np.sqrt(mean_squared_error(y_true_train_unscaled, y_pred_train_unscaled))
mae_train_unscaled = mean_absolute_error(y_true_train_unscaled, y_pred_train_unscaled)
r2_train_unscaled = r2_score(y_true_train_unscaled, y_pred_train_unscaled)

# ========== Validation Unscaled Metrics ==========
y_true_val_scaled = np.array(val_df[target_col].values).reshape(-1, 1)

val_dataset_noscale = ChlCNNMLPTemporalMLPFusionDataset(
    val_df, spatial_cols, temporal_cols, target_col, augment=False
)
val_loader_noscale = DataLoader(val_dataset_noscale, batch_size=HYPERPARAMS['batch_size'], shuffle=False)

y_pred_val_scaled = []
with torch.no_grad():
    for x_spat, x_temp, _ in val_loader_noscale:
        x_spat, x_temp = x_spat.to(HYPERPARAMS['device']), x_temp.to(HYPERPARAMS['device'])
        y_pred = model(x_spat, x_temp)
        y_pred_val_scaled.append(y_pred.cpu().numpy())
y_pred_val_scaled = np.concatenate(y_pred_val_scaled)

y_true_val_unscaled = target_scl.inverse_transform(y_true_val_scaled).ravel()
y_pred_val_unscaled = target_scl.inverse_transform(y_pred_val_scaled.reshape(-1, 1)).ravel()

rmse_val_unscaled = np.sqrt(mean_squared_error(y_true_val_unscaled, y_pred_val_unscaled))
mae_val_unscaled = mean_absolute_error(y_true_val_unscaled, y_pred_val_unscaled)
r2_val_unscaled = r2_score(y_true_val_unscaled, y_pred_val_unscaled)

train_metrics = [rmse_train_unscaled, mae_train_unscaled, r2_train_unscaled]
val_metrics   = [rmse_val_unscaled,  mae_val_unscaled,  r2_val_unscaled]

# --- Print Results ---
print("\n========== Final Training Set Metrics (Unscaled mg/m³) ==========")
print(f"RMSE | Train: {rmse_train_unscaled:.4f} | Val: {rmse_val_unscaled:.4f}")
print(f"MAE  | Train: {mae_train_unscaled:.4f} | Val: {mae_val_unscaled:.4f}")
print(f"R²   | Train: {r2_train_unscaled:.4f} | Val: {r2_val_unscaled:.4f}")



# ========== 10. TEST SET EVALUATION ==========


# --- Prepare test set ---
test_dataset = ChlCNNMLPTemporalMLPFusionDataset(
    test_df, spatial_cols, temporal_cols, target_col,
    augment=False
)
test_loader = DataLoader(
    test_dataset, batch_size=HYPERPARAMS['batch_size'], shuffle=False, drop_last=False
)

# --- Load final model ---
model = CNNSpatial_MLPTemporal_MLPFusion(
    n_temp_features=len(temporal_cols),
    h_c=HYPERPARAMS['h_c'],
    drop=HYPERPARAMS['drop']
)
model.load_state_dict(torch.load("cnn_mlp_mlp_final.pt", map_location=HYPERPARAMS['device']))
model.to(HYPERPARAMS['device'])
model.eval()

# --- Evaluate on test set (scaled) ---
y_true, y_pred = [], []
with torch.no_grad():
    for x_spat, x_temp, y in test_loader:
        x_spat, x_temp = x_spat.to(HYPERPARAMS['device']), x_temp.to(HYPERPARAMS['device'])
        outputs = model(x_spat, x_temp)
        y_true.append(y.numpy())
        y_pred.append(outputs.cpu().numpy())
y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

# --- Inverse transform to original scale (mg/m³) ---
y_true_unscaled = target_scl.inverse_transform(y_true.reshape(-1, 1)).ravel()
y_pred_unscaled = target_scl.inverse_transform(y_pred.reshape(-1, 1)).ravel()

# --- Compute metrics in original units ---
rmse_test = np.sqrt(mean_squared_error(y_true_unscaled, y_pred_unscaled))
mae_test = mean_absolute_error(y_true_unscaled, y_pred_unscaled)
r2_test = r2_score(y_true_unscaled, y_pred_unscaled)

print(f"\n========== Final Test Set Metrics (mg/m³) ==========")
print(f"RMSE: {rmse_test:.4f} mg/m³")
print(f"MAE : {mae_test:.4f} mg/m³")
print(f"R²  : {r2_test:.4f}")

# ----------------- Scatter Plot -------------------
plt.figure(figsize=(6,6))
plt.scatter(y_true_unscaled, y_pred_unscaled, alpha=0.25, s=8, edgecolor='none')
plt.plot([y_true_unscaled.min(), y_true_unscaled.max()],
         [y_true_unscaled.min(), y_true_unscaled.max()],
         'r--', lw=2, label='1:1')
plt.xlabel('True (unScaled)')
plt.ylabel('Predicted (unScaled)')
plt.title('Test Set: True vs. Predicted')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# ------------- Bar Chart ----------------
metrics = ['RMSE', 'MAE', 'R²']
train_vals = train_metrics
val_vals   = val_metrics
test_vals  = [rmse_test, mae_test, r2_test]

fig = go.Figure(data=[
    go.Bar(name='Train', x=metrics, y=train_vals, marker_color='rgb(66, 133, 244)'),
    go.Bar(name='Val',   x=metrics, y=val_vals,   marker_color='rgb(244, 180, 0)'),
    go.Bar(name='Test',  x=metrics, y=test_vals,  marker_color='rgb(219, 68, 55)')
])
fig.update_layout(
    barmode='group',
    title='Train vs. Val vs. Test Metrics (Single Model)',
    yaxis_title='Metric Value',
    xaxis_title='Metric',
    template='plotly_white',
    legend=dict(x=0.6, y=1.1, orientation='h')
)
fig.show()


# ========== 11. GAP FILLING  ==========


gap_df = df[(df['mask'] == 1) & (df['nbr_valid_count'] <= 4)].dropna(subset=feature_cols).copy()
print(f"Total gaps to fill: {len(gap_df)}")

# NOTE: spatial_cols contains all 7×9 = 63 features as 7 spatial channels
gap_dataset = ChlCNNMLPTemporalMLPFusionDataset(
    gap_df, spatial_cols, temporal_cols, target_col,
    augment=False
)
gap_loader = DataLoader(
    gap_dataset,
    batch_size=HYPERPARAMS['batch_size'],
    shuffle=False,
    drop_last=False
)

# --- Load trained model ---
model = CNNSpatial_MLPTemporal_MLPFusion(
    n_temp_features=len(temporal_cols),
    h_c=HYPERPARAMS['h_c'],
    drop=HYPERPARAMS['drop']
)
model.load_state_dict(torch.load("cnn_mlp_mlp_final.pt", map_location=HYPERPARAMS['device']))
model.to(HYPERPARAMS['device'])
model.eval()

# --- Predict gap values ---
preds = []
with torch.no_grad():
    for x_spat, x_temp, _ in gap_loader:
        x_spat, x_temp = x_spat.to(HYPERPARAMS['device']), x_temp.to(HYPERPARAMS['device'])
        y_pred = model(x_spat, x_temp)
        preds.append(y_pred.cpu().numpy())

gap_preds = np.concatenate(preds)
gap_preds_unscaled = target_scl.inverse_transform(gap_preds.reshape(-1, 1)).ravel()
gap_df['chl_gapfill'] = gap_preds_unscaled

# --- Update master DataFrame ---
df['chl_gapfill'] = df['chl_orig']
df.loc[gap_df.index, 'chl_gapfill'] = gap_df['chl_gapfill']

# --- Create xarray output ---
chl_gapfill_xr = xr.DataArray(
    np.full_like(chl_clean.values, np.nan, dtype=np.float32),
    dims=chl_clean.dims,
    coords=chl_clean.coords
)
# Create fast lookup indices for time, lat, lon
time_indexer = {t: i for i, t in enumerate(chl_gapfill_xr['time'].values)}
lat_indexer  = {v: i for i, v in enumerate(chl_gapfill_xr['lat'].values)}
lon_indexer  = {v: i for i, v in enumerate(chl_gapfill_xr['lon'].values)}

# Convert rows to arrays
time_idx = df['time'].map(time_indexer).values
lat_idx  = df['lat'].map(lat_indexer).values
lon_idx  = df['lon'].map(lon_indexer).values

# Fill in all at once
chl_gapfill_xr.values[time_idx, lat_idx, lon_idx] = df['chl_gapfill'].values

print(f"✅ Gap-filling complete! {len(gap_df)} missing points filled using final model.")
print(f"chl_gapfill_xr shape: {chl_gapfill_xr.shape} (xarray DataArray)")


# ========== 12. CONTOUR VISUALIZATION (Observed vs. Gap-filled, Random 3 Days, 35–45% observed) ==========

import plotly.subplots as sp
import plotly.graph_objs as go



custom_scale = [
    [0.0, 'blue'],      # Low values
    [0.25, 'cyan'],     # Light blue
    [0.5, 'yellow'],    # Mid values
    [0.75, 'orange'],   # Higher values
    [1.0, 'red']        # Highest values
]

# --- 1. Compute observation percentages ---
all_times = pd.to_datetime(chl_gapfill_xr['time'].values)
obs_ratio = {}
for t in all_times:
    day_data = chl_clean.sel(time=t).values
    total_points = day_data.size
    valid_points = np.count_nonzero(~np.isnan(day_data))
    obs_ratio[t] = (valid_points / total_points) * 100  # percentage

# Filter days with 35–45% observed data
filtered_days = [t for t, pct in obs_ratio.items() if 35 <= pct <= 45]

# If not enough days are found, fallback to any missing-data days
has_missing = [t for t in all_times if np.isnan(chl_clean.sel(time=t).values).any()]

np.random.seed(42)
if len(filtered_days) >= 3:
    chosen_days = np.random.choice(filtered_days, 3, replace=False)
elif len(filtered_days) > 0:
    chosen_days = np.random.choice(filtered_days, min(3, len(filtered_days)), replace=False)
else:
    chosen_days = np.random.choice(has_missing, 3, replace=False)

# --- 2. Plot Observed vs. Gap-filled for chosen days ---
for i, sel_day in enumerate(chosen_days, 1):
    observed = chl_clean.sel(time=sel_day).values
    filled   = chl_gapfill_xr.sel(time=sel_day).values

    fig = sp.make_subplots(
        rows=1, cols=2, shared_yaxes=True, shared_xaxes=True,
        subplot_titles=(f"Observed Chl-a ({obs_ratio[sel_day]:.1f}% obs)", "Gap-filled Chl-a")
    )
    # --- Observed ---
    fig.add_trace(go.Contour(
        z=observed,
        colorscale=custom_scale,
        colorbar=dict(title="mg/m³", x=0.46, len=0.75),
        contours=dict(showlines=False),
        zmin=np.nanmin(filled), zmax=np.nanmax(filled),
        hovertemplate="Lat: %{y}<br>Lon: %{x}<br>Chl-a: %{z:.3f} mg/m³<extra></extra>",
    ), row=1, col=1)
    # --- Gap-filled ---
    fig.add_trace(go.Contour(
        z=filled,
        colorscale=custom_scale,
        colorbar=dict(title="mg/m³", x=1.02, len=0.75),
        contours=dict(showlines=False),
        zmin=np.nanmin(filled), zmax=np.nanmax(filled),
        hovertemplate="Lat: %{y}<br>Lon: %{x}<br>Chl-a: %{z:.3f} mg/m³<extra></extra>",
    ), row=1, col=2)

    fig.update_layout(
        height=400, width=950,
        title_text=f"Observed vs. Gap-filled Chl-a on {str(sel_day)[:10]}",
        template="plotly_white"
    )
    fig.update_xaxes(title_text="Longitude", row=1, col=1)
    fig.update_xaxes(title_text="Longitude", row=1, col=2)
    fig.update_yaxes(title_text="Latitude", row=1, col=1)
    fig.show()

